import des éléments 

In [18]:
from IPython.display import Image, display
from urllib.parse import urljoin
import pandas as pd
import requests
import sqlite3
from bs4 import BeautifulSoup

url du site à scrper

In [8]:
url = "https://www.allocine.fr/film/aucinema/"
response = requests.get(url, timeout=10)
response.raise_for_status()
print("Code HTTP :", response.status_code)

Code HTTP : 200


Soup

In [19]:
soup = BeautifulSoup(response.text, "html.parser")
films = soup.select("div.card.entity-card")
def recuperer_texte(film, selecteur):
    element = film.select_one(selecteur)
    return element.get_text(" ", strip=True) if element else None
#print("Nombre d'éléments :", films)


**Extraction des informations demandées**

In [25]:
donnees = []

for film in films:
    titre_element = film.select_one("h2.meta-title a")

    if titre_element is None:
        continue

    titre = titre_element.get_text(" ", strip=True)
    lien = urljoin(url, titre_element["href"])

    synopsis = recuperer_texte(film, "div.content-txt")
    informations = recuperer_texte(film, "div.meta-body-info")
    acteurs = recuperer_texte(film, "div.meta-body-actor")

    # Séparation de la date, de la durée et des genres
    parties = [partie.strip() for partie in informations.split("|")] if informations else []

    date_sortie = parties[0]
    duree = parties[1] 
    genres = parties[2] 

    notes = [
        element.get_text(" ", strip=True)
        for element in film.select("span.stareval-note")
    ]


    note_presse = notes[0]
    note_spectateurs = notes[1] 

    donnees.append({
        "titre": titre,
        "lien": lien,
        "synopsis": synopsis,
        "date_sortie": date_sortie,
        "duree": duree,
        "genres": genres,
        "acteurs": acteurs,
        "note_presse": note_presse,
        "note_spectateurs": note_spectateurs
    })

df = pd.DataFrame(donnees)

display(df.head(10))

,titre,lien,synopsis,date_sortie,duree,genres,acteurs,note_presse,note_spectateurs
0,La Vénus électrique,https://www.allocine.fr/film/fichefilm_gen_cfi...,"Paris, 1928. Antoine Balestro, jeune peintre e...",12 mai 2026,2h 02min,"Comédie , Romance","Avec Pio Marmaï , Anaïs Demoustier , Gilles Le...","4,0","3,9"
1,Le Diable s'habille en Prada 2,https://www.allocine.fr/film/fichefilm_gen_cfi...,"Miranda, Andy, Emily et Nigel replongent dans ...",29 avril 2026,1h 59min,"Comédie , Drame","Avec Meryl Streep , Anne Hathaway , Emily Blunt","2,9","3,2"
2,Pour le plaisir,https://www.allocine.fr/film/fichefilm_gen_cfi...,Et si on vous racontait l’invention du siècle ...,6 mai 2026,1h 29min,Comédie,"Avec Alexandra Lamy , François Cluzet , Mitty ...","3,1","3,8"
3,Star Wars: The Mandalorian and Grogu,https://www.allocine.fr/film/fichefilm_gen_cfi...,La chute du maléfique Empire Galactique a préc...,20 mai 2026,2h 12min,"Action , Aventure , Fantastique","Avec Pedro Pascal , Sigourney Weaver , Jeremy ...","3,0","3,4"
4,Michael,https://www.allocine.fr/film/fichefilm_gen_cfi...,Michael dresse le portrait cinématographique d...,22 avril 2026,2h 08min,"Biopic , Drame , Musical","Avec Jaafar Jackson , Colman Domingo , Nia Long","2,4","4,1"
5,L’Abandon,https://www.allocine.fr/film/fichefilm_gen_cfi...,"Le 16 octobre 2020, Samuel Paty, professeur d’...",13 mai 2026,1h 40min,Drame,"Avec Antoine Reinartz , Emmanuelle Bercot , Em...","3,5","4,4"
6,Histoires parallèles,https://www.allocine.fr/film/fichefilm_gen_cfi...,"En quête d’inspiration pour son nouveau roman,...",14 mai 2026,2h 19min,Drame,"Avec Isabelle Huppert , Virginie Efira , Pierr...","3,3","2,9"
7,L'Objet du délit,https://www.allocine.fr/film/fichefilm_gen_cfi...,Dans les coulisses d'une ambitieuse production...,27 mai 2026,2h 13min,Comédie dramatique,"Avec Agnès Jaoui , Daniel Auteuil , Eye Haïdara","3,3","3,5"
8,C’est quoi l’amour ?,https://www.allocine.fr/film/fichefilm_gen_cfi...,"Lorsque Fred demande à son ex-femme, Marguerit...",6 mai 2026,1h 48min,Comédie,"Avec Laure Calamy , Vincent Macaigne , Lyes Salem","3,8","3,7"
9,Autofiction,https://www.allocine.fr/film/fichefilm_gen_cfi...,Raúl est un cinéaste culte en pleine crise c...,20 mai 2026,1h 51min,Comédie dramatique,"Avec Bárbara Lennie , Leonardo Sbaraglia , Ait...","3,6","2,8"


**Persistance des données**

In [26]:
connexion = sqlite3.connect("allocine.db")

df.to_sql(
    name="films",
    con=connexion,
    if_exists="replace",
    index=False
)

connexion.close()

print("Données enregistrées dans allocine.db")

Données enregistrées dans allocine.db


**Pour vérifier que la base contient bien tes données :**

In [27]:
connexion = sqlite3.connect("allocine.db")
curseur = connexion.cursor()

curseur.execute("SELECT COUNT(*) FROM films")
print("Nombre de films :", curseur.fetchone()[0])

connexion.close()

Nombre de films : 15


**Pour relire et afficher les données :**

In [28]:
connexion = sqlite3.connect("allocine.db")

films_db = pd.read_sql_query(
    "SELECT * FROM films",
    connexion
)

connexion.close()
films_db.head(10)

,titre,lien,synopsis,date_sortie,duree,genres,acteurs,note_presse,note_spectateurs
0,La Vénus électrique,https://www.allocine.fr/film/fichefilm_gen_cfi...,"Paris, 1928. Antoine Balestro, jeune peintre e...",12 mai 2026,2h 02min,"Comédie , Romance","Avec Pio Marmaï , Anaïs Demoustier , Gilles Le...","4,0","3,9"
1,Le Diable s'habille en Prada 2,https://www.allocine.fr/film/fichefilm_gen_cfi...,"Miranda, Andy, Emily et Nigel replongent dans ...",29 avril 2026,1h 59min,"Comédie , Drame","Avec Meryl Streep , Anne Hathaway , Emily Blunt","2,9","3,2"
2,Pour le plaisir,https://www.allocine.fr/film/fichefilm_gen_cfi...,Et si on vous racontait l’invention du siècle ...,6 mai 2026,1h 29min,Comédie,"Avec Alexandra Lamy , François Cluzet , Mitty ...","3,1","3,8"
3,Star Wars: The Mandalorian and Grogu,https://www.allocine.fr/film/fichefilm_gen_cfi...,La chute du maléfique Empire Galactique a préc...,20 mai 2026,2h 12min,"Action , Aventure , Fantastique","Avec Pedro Pascal , Sigourney Weaver , Jeremy ...","3,0","3,4"
4,Michael,https://www.allocine.fr/film/fichefilm_gen_cfi...,Michael dresse le portrait cinématographique d...,22 avril 2026,2h 08min,"Biopic , Drame , Musical","Avec Jaafar Jackson , Colman Domingo , Nia Long","2,4","4,1"
5,L’Abandon,https://www.allocine.fr/film/fichefilm_gen_cfi...,"Le 16 octobre 2020, Samuel Paty, professeur d’...",13 mai 2026,1h 40min,Drame,"Avec Antoine Reinartz , Emmanuelle Bercot , Em...","3,5","4,4"
6,Histoires parallèles,https://www.allocine.fr/film/fichefilm_gen_cfi...,"En quête d’inspiration pour son nouveau roman,...",14 mai 2026,2h 19min,Drame,"Avec Isabelle Huppert , Virginie Efira , Pierr...","3,3","2,9"
7,L'Objet du délit,https://www.allocine.fr/film/fichefilm_gen_cfi...,Dans les coulisses d'une ambitieuse production...,27 mai 2026,2h 13min,Comédie dramatique,"Avec Agnès Jaoui , Daniel Auteuil , Eye Haïdara","3,3","3,5"
8,C’est quoi l’amour ?,https://www.allocine.fr/film/fichefilm_gen_cfi...,"Lorsque Fred demande à son ex-femme, Marguerit...",6 mai 2026,1h 48min,Comédie,"Avec Laure Calamy , Vincent Macaigne , Lyes Salem","3,8","3,7"
9,Autofiction,https://www.allocine.fr/film/fichefilm_gen_cfi...,Raúl est un cinéaste culte en pleine crise c...,20 mai 2026,1h 51min,Comédie dramatique,"Avec Bárbara Lennie , Leonardo Sbaraglia , Ait...","3,6","2,8"
